In [311]:
import pandas as pd
import numpy as np
from function_file import standardize_columns, normalize_data, one_hot_encoding
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingRegressor, BaggingClassifier,RandomForestRegressor,AdaBoostRegressor, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.preprocessing import OneHotEncoder

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [312]:
df = pd.read_csv("survey.csv")
print(df.head(5))

             Timestamp  Age  Gender         Country state self_employed  \
0  2014-08-27 11:29:31   37  Female   United States    IL           NaN   
1  2014-08-27 11:29:37   44       M   United States    IN           NaN   
2  2014-08-27 11:29:44   32    Male          Canada   NaN           NaN   
3  2014-08-27 11:29:46   31    Male  United Kingdom   NaN           NaN   
4  2014-08-27 11:30:22   31    Male   United States    TX           NaN   

  family_history treatment work_interfere    no_employees  ...  \
0             No       Yes          Often            6-25  ...   
1             No        No         Rarely  More than 1000  ...   
2             No        No         Rarely            6-25  ...   
3            Yes       Yes          Often          26-100  ...   
4             No        No          Never         100-500  ...   

                leave mental_health_consequence phys_health_consequence  \
0       Somewhat easy                        No                      No   
1 

In [313]:
df.duplicated().sum()

np.int64(0)

In [314]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1259 entries, 0 to 1258
Data columns (total 27 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   Timestamp                  1259 non-null   object
 1   Age                        1259 non-null   int64 
 2   Gender                     1259 non-null   object
 3   Country                    1259 non-null   object
 4   state                      744 non-null    object
 5   self_employed              1241 non-null   object
 6   family_history             1259 non-null   object
 7   treatment                  1259 non-null   object
 8   work_interfere             995 non-null    object
 9   no_employees               1259 non-null   object
 10  remote_work                1259 non-null   object
 11  tech_company               1259 non-null   object
 12  benefits                   1259 non-null   object
 13  care_options               1259 non-null   object
 14  wellness

In [315]:
df.shape

(1259, 27)

In [316]:
df.columns

Index(['Timestamp', 'Age', 'Gender', 'Country', 'state', 'self_employed',
       'family_history', 'treatment', 'work_interfere', 'no_employees',
       'remote_work', 'tech_company', 'benefits', 'care_options',
       'wellness_program', 'seek_help', 'anonymity', 'leave',
       'mental_health_consequence', 'phys_health_consequence', 'coworkers',
       'supervisor', 'mental_health_interview', 'phys_health_interview',
       'mental_vs_physical', 'obs_consequence', 'comments'],
      dtype='object')

- For this project I would b econsidering Treatment as Target variable
- There are some columns which are related to this column directl, so i will drop these columns
- also as many columns present to be considered as feature but i for project to keep simple i am 

In [317]:
# Updated for Tech Survey Dataset
selected_columns = [
    # Your latest requested columns (Stigma & Culture)
    'phys_health_consequence',
    'coworkers',
    
    # HIGH IMPACT columns (Essential for getting above 0.33 accuracy)
    'family_history',
    'benefits',
    'Age',
    'work_interfere',
    
    # Target column (In this dataset, 'treatment' is the goal)
    'treatment'  
]

# Apply the selection to your new dataframe
df = df[selected_columns]

# View remaining columns
print("Columns successfully updated for Tech Survey Analysis:")
print(df.columns)

Columns successfully updated for Tech Survey Analysis:
Index(['phys_health_consequence', 'coworkers', 'family_history', 'benefits',
       'Age', 'work_interfere', 'treatment'],
      dtype='object')


In [318]:
# Datatypes of columns
df.dtypes

phys_health_consequence    object
coworkers                  object
family_history             object
benefits                   object
Age                         int64
work_interfere             object
treatment                  object
dtype: object

Checking for missing values

In [319]:
((df.isna().sum())/len(df))*100

phys_health_consequence     0.000000
coworkers                   0.000000
family_history              0.000000
benefits                    0.000000
Age                         0.000000
work_interfere             20.969023
treatment                   0.000000
dtype: float64

**Perform train test**

In [320]:
features = df.drop(columns=['treatment'])
target = df["treatment"]

In [321]:
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.20, random_state=17)

In [322]:
# 1. Standardize and Map the Target (treatment)
y_train = y_train.map({'Yes': 1, 'No': 0})
y_test = y_test.map({'Yes': 1, 'No': 0})

In [323]:
print(X_train.shape)
print(X_test.shape)

(1007, 6)
(252, 6)


In [324]:
X_train

,phys_health_consequence,coworkers,family_history,benefits,Age,work_interfere
132,No,Yes,No,Don't know,27,Never
909,Maybe,Yes,No,Yes,48,Never
744,Maybe,Some of them,No,Yes,36,Sometimes
1012,Maybe,Some of them,Yes,Yes,24,Sometimes
1257,No,No,No,No,46,NaN
...,...,...,...,...,...,...
278,No,Some of them,No,Yes,28,Rarely
752,No,Some of them,Yes,Don't know,37,Sometimes
406,Maybe,Yes,Yes,Yes,33,Never
143,No,Some of them,No,Yes,-29,NaN


In [325]:
# X_train_encoded = standardize_columns(X_train)
encoder = OneHotEncoder(sparse_output = False) #initialize the function in skilearn
encoder.fit(X_train[["work_interfere","family_history","benefits","phys_health_consequence","coworkers"]])
X_train_encoded = one_hot_encoding(X_train,encoder)

In [326]:
X_train_encoded

,Age,work_interfere_Never,work_interfere_Often,work_interfere_Rarely,work_interfere_Sometimes,work_interfere_nan,family_history_No,family_history_Yes,benefits_Don't know,benefits_No,benefits_Yes,phys_health_consequence_Maybe,phys_health_consequence_No,phys_health_consequence_Yes,coworkers_No,coworkers_Some of them,coworkers_Yes
0,27,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
1,48,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0
2,36,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0
3,24,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0
4,46,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1002,28,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
1003,37,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
1004,33,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0
1005,-29,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0


Doing normalization so that all features are in one range

In [327]:
normalizer = MinMaxScaler()

In [328]:
X_train_encoded

,Age,work_interfere_Never,work_interfere_Often,work_interfere_Rarely,work_interfere_Sometimes,work_interfere_nan,family_history_No,family_history_Yes,benefits_Don't know,benefits_No,benefits_Yes,phys_health_consequence_Maybe,phys_health_consequence_No,phys_health_consequence_Yes,coworkers_No,coworkers_Some of them,coworkers_Yes
0,27,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
1,48,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0
2,36,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0
3,24,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0
4,46,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1002,28,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
1003,37,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
1004,33,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0
1005,-29,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0


In [329]:
normalizer.fit(X_train_encoded)

,feature_range,"(0, ...)"
,copy,True
,clip,False


In [330]:
X_train_norm = normalize_data(X_train_encoded, normalizer)


In [331]:
X_train_norm

,Age,work_interfere_Never,work_interfere_Often,work_interfere_Rarely,work_interfere_Sometimes,work_interfere_nan,family_history_No,family_history_Yes,benefits_Don't know,benefits_No,benefits_Yes,phys_health_consequence_Maybe,phys_health_consequence_No,phys_health_consequence_Yes,coworkers_No,coworkers_Some of them,coworkers_Yes
0,1.753000e-08,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
1,1.774000e-08,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0
2,1.762000e-08,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0
3,1.750000e-08,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0
4,1.772000e-08,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1002,1.754000e-08,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
1003,1.763000e-08,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
1004,1.759000e-08,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0
1005,1.697000e-08,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0


In [332]:
from sklearn.neighbors import KNeighborsClassifier
model = KNeighborsClassifier(n_neighbors = 10)

In [333]:
X_train_norm.shape

(1007, 17)

In [334]:
model.fit(X_train_norm,y_train)

,n_neighbors,10
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [335]:
#X_test = one_hot_encoding(X_test)
#X_test_stand= standardize_columns(X_test)
X_test_stand = one_hot_encoding(X_test,encoder)
X_test_norm = normalize_data(X_test_stand, normalizer)


In [336]:
round(model.score(X_test_norm, y_test),4)

0.7738

Trying liniar regression

In [337]:
lin_reg = LinearRegression()

In [338]:
lin_reg.fit(X_train_norm, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [339]:
round(lin_reg.score(X_test_norm, y_test),4)

0.4763

In [340]:
from sklearn.ensemble import RandomForestClassifier

# Initialize and fit
df = RandomForestClassifier(n_estimators=100, random_state=17)
df.fit(X_train_norm, y_train)

# Score
print(round(df.score(X_test_norm, y_test),4))

0.7976


In [341]:
tree = DecisionTreeClassifier(max_depth=10)

In [342]:
tree.fit(X_train_norm, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,10
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [343]:
round(tree.score(X_test_norm,y_test),4)

0.7619

In [344]:
df_imp = pd.DataFrame({
    "features": X_test_norm.columns,
    'impoertance': tree.feature_importances_
})

df_imp

,features,impoertance
0,Age,0.168427
1,work_interfere_Never,0.294525
2,work_interfere_Often,0.011284
3,work_interfere_Rarely,0.019074
4,work_interfere_Sometimes,0.015432
5,work_interfere_nan,0.340816
6,family_history_No,0.001363
7,family_history_Yes,0.031583
8,benefits_Don't know,0.012861
9,benefits_No,0.012692


Bagging and Pasting

In [345]:
bagging_reg = BaggingClassifier(DecisionTreeClassifier(max_depth=20),
                               n_estimators=100,
                               max_samples = 1000)

In [346]:
bagging_reg.fit(X_train_norm, y_train)

,estimator,DecisionTreeC...(max_depth=20)
,n_estimators,100
,max_samples,1000
,max_features,1.0
,bootstrap,True
,bootstrap_features,False
,oob_score,False
,warm_start,False
,n_jobs,None
,random_state,None
,verbose,0


In [347]:
forest = RandomForestClassifier(n_estimators=100,
                             max_depth=20)

In [348]:
pred = bagging_reg.predict(X_test_norm)
print("R2 score", round(bagging_reg.score(X_test_norm, y_test),4))

R2 score 0.7778


Random Patches

In [349]:
forest = RandomForestClassifier(n_estimators=100,
                             max_depth=20)
forest.fit(X_train_norm, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,20
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [350]:
pred = forest.predict(X_test_norm)
print("R2 score", round(forest.score(X_test_norm, y_test),4))

R2 score 0.8056


Adaboost

In [351]:
ada_reg = AdaBoostClassifier(DecisionTreeClassifier(max_depth=20),
                            n_estimators=100)
ada_reg.fit(X_train_norm, y_train)

,estimator,DecisionTreeC...(max_depth=20)
,n_estimators,100
,learning_rate,1.0
,algorithm,'deprecated'
,random_state,None
,criterion,'gini'
,splitter,'best'
,max_depth,20
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0


In [352]:
pred = ada_reg.predict(X_test_norm)

print("R2 score", round(ada_reg.score(X_test_norm, y_test),4))

R2 score 0.7698


Gradient Boosting

In [353]:
gb_reg = GradientBoostingClassifier(max_depth=20,
                                   n_estimators=100)
gb_reg.fit(X_train_norm, y_train)

,loss,'log_loss'
,learning_rate,0.1
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,20
,min_impurity_decrease,0.0
,init,None


In [354]:
pred = gb_reg.predict(X_test_norm)
print("R2 score", round(gb_reg.score(X_test_norm, y_test),4))

R2 score 0.7619


In [355]:
import joblib

# 'model' is your trained DecisionTreeClassifier
joblib.dump(forest, 'tech_survey_model.pkl')
print("Model saved successfully!")

Model saved successfully!


In [356]:
# 'normalizer' is the name of the MinMaxScaler you fitted in your notebook
joblib.dump(normalizer, 'scaler.pkl')

print("Success! 'scaler.pkl' has been created in your project folder.")

Success! 'scaler.pkl' has been created in your project folder.


In [357]:
# 'encoder' is the name of the MinMaxScaler you fitted in your notebook
joblib.dump(encoder, 'encoder.pkl')

print("Success! 'encoder.pkl' has been created in your project folder.")

Success! 'encoder.pkl' has been created in your project folder.


In [358]:
# This prints the EXACT order the model learned
print("Order for Streamlit:", X_train.columns.tolist())

Order for Streamlit: ['phys_health_consequence', 'coworkers', 'family_history', 'benefits', 'Age', 'work_interfere']
